In [0]:
import os
import re
from pyspark.sql.functions import current_timestamp
from pyspark.sql.functions import col
from pyspark.sql.types import StringType
from pyspark.sql.functions import regexp_replace, col

CLEAR_RAW = True # True or False

def pascal_to_upper_snake(name: str) -> str:
    # Insere "_" antes de letras maiúsculas (exceto na primeira posição) e converte tudo para UPPERCASE
    return re.sub(r'(?<!^)(?=[A-Z])', '_', name).upper()

def drop_tables(tables, schema="adventure_works.raw"):
    for name in tables:
        query = f"DROP TABLE IF EXISTS {schema}.{name}"
        spark.sql(query)

# api adventure_works/raw_extract
dir_api = "/Volumes/adventure_works/raw_extract/raw_api"


files_api = [file for file in os.listdir(dir_api)]
tables_api = [pascal_to_upper_snake(file.split('.')[0]) for file in files_api]
dir_tables_api = [dir_api+'/'+file for file in files_api]

# db
dir_db = "/Volumes/adventure_works/raw_extract/raw_db"


files_db = [file for file in os.listdir(dir_db)]
tables_db = [pascal_to_upper_snake(file.split('.')[0]) for file in files_db]
dir_tables_db = [f"{dir_db}/{file}" for file in files_db]

all_tables = tables_api + tables_db
list_to_delta = dir_tables_api + dir_tables_db
files = files_api + files_db

if CLEAR_RAW:
    drop_tables(all_tables)

for file, abs_path in zip(files, list_to_delta):

    df_bronze = (
        spark.read
        .format("json")
        .option("multiLine", "false")
        .load(abs_path)
        )
    
    df_bronze = df_bronze.withColumn("load_at", current_timestamp())

    df_bronze = df_bronze.select([col(c).cast(StringType()).alias(c) for c in df_bronze.columns])

    save_delta_name = pascal_to_upper_snake(file.split('.')[0])
    print(save_delta_name.upper())

    # === sanitização das colunas ID ===
    df_bronze = df_bronze.select([
        regexp_replace(col(c), r'\.0+$', '').alias(c) 
        if 'id ' in c.lower() else col(c)
        for c in df_bronze.columns
    ])

    # Escrita no formato Delta para a camada RAW opção - > .mode("append")\
    df_bronze.write\
        .option("overwriteSchema", "true")\
        .format("delta")\
        .mode("overwrite")\
        .saveAsTable(f"adventure_works.raw.{save_delta_name.upper()}")


PURCHASE_ORDER_DETAIL
PURCHASE_ORDER_HEADER
SALES_ORDER_DETAIL
SALES_ORDER_HEADER
ADDRESS
ADDRESS_TYPE
BUSINESS_ENTITY
BUSINESS_ENTITY_ADDRESS
BUSINESS_ENTITY_CONTACT
CONTACT_TYPE
COUNTRY_REGION
COUNTRY_REGION_CURRENCY
CREDIT_CARD
CULTURE
CURRENCY
CURRENCY_RATE
CUSTOMER
DEPARTMENT
EMAIL_ADDRESS
EMPLOYEE
LOCATION
PERSON
PERSON_CREDIT_CARD
PERSON_PHONE
PHONE_NUMBER_TYPE
PRODUCT
PRODUCT_CATEGORY
PRODUCT_COST_HISTORY
PRODUCT_INVENTORY
PRODUCT_LIST_PRICE_HISTORY
PRODUCT_MODEL
PRODUCT_MODEL_PRODUCT_DESCRIPTION_CULTURE
PRODUCT_SUBCATEGORY
PRODUCT_VENDOR
PURCHASE_ORDER_DETAIL
PURCHASE_ORDER_HEADER
SALES_ORDER_DETAIL
SALES_ORDER_HEADER
SALES_ORDER_HEADER_SALES_REASON
SALES_PERSON
SALES_PERSON_QUOTA_HISTORY
SALES_REASON
SALES_TAX_RATE
SALES_TERRITORY
SALES_TERRITORY_HISTORY
SCRAP_REASON
SHIP_METHOD
SHOPPING_CART_ITEM
SPECIAL_OFFER
SPECIAL_OFFER_PRODUCT
STATE_PROVINCE
STORE
TRANSACTION_HISTORY
TRANSACTION_HISTORY_ARCHIVE
VENDOR
WORK_ORDER
WORK_ORDER_ROUTING
